In [ ]:
from pathlib import Path
import pandas as pd

# Find all CSV files 
# Path(".").glob("*/*.csv")
csv_files = list(Path(".").rglob("*/*.csv"))

print(f"Found {len(csv_files)} CSV files")

# Read and merge
df_list = []

for file in csv_files:
    print(f"Reading: {file}")
    df = pd.read_csv(file)
    
    # Optional: keep source file name
    df["source_file"] = str(file)
    
    df_list.append(df)

# Combine all CSV files
merged_df = pd.concat(df_list, ignore_index=True)

# Save merged file
merged_df.to_excel("merged_all_csv_files.xlsx", index=False)

print("Merged file saved as: merged_all_csv_files.csv")
print("Final shape:", merged_df.shape)

In [ ]:
# # Distinct values of os_architecture
# distinct_os_architecture = merged_df["os_architecture"].dropna().unique()

# print(distinct_os_architecture)

In [ ]:
# cpu_model = merged_df["cpu_model"].dropna().unique()
# print(cpu_model)

In [ ]:
side_channel_features = [
    "execution_time_sec",
    "cpu_energy_kwh",
    "gpu_energy_kwh",
    "ram_energy_kwh",
    "total_energy_kwh",
    "total_emissions_kg",
    "joules_per_token",
    "energy_per_token_kwh",
    "watts_estimated",
    "tokens_per_second",
    "cpu_usage_pct",
    "cpu_clock_mhz",
    "cpu_cores_used",
    "ram_usage_pct",
    "memory_footprint_mb"
]


targets = [
    "circuit_type",
    "n_qubits",
    "n_layers",
    "model_type"
]

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


# =========================
# 1. Define features/targets
# =========================

side_channel_features = [
    "execution_time_sec",
    "cpu_energy_kwh",
    "gpu_energy_kwh",
    "ram_energy_kwh",
    "total_energy_kwh",
    "total_emissions_kg",
    "joules_per_token",
    "energy_per_token_kwh",
    "watts_estimated",
    "tokens_per_second",
    "cpu_usage_pct",
    "cpu_clock_mhz",
    "cpu_cores_used",
    "ram_usage_pct",
    "memory_footprint_mb"
]

targets = [
    "circuit_type",
    "n_qubits",
    "n_layers",
   # "model_type"
]


# =========================
# 2. Clean available columns
# =========================

available_features = [col for col in side_channel_features if col in merged_df.columns]
available_targets = [col for col in targets if col in merged_df.columns]

print("Available side-channel features:")
print(available_features)

print("\nAvailable target columns:")
print(available_targets)


# =========================
# 3. Define classifiers
# =========================

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]),

    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(random_state=42))
    ]),

    "SVM RBF": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=True,
            random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ])
}


# =========================
# 4. Train/evaluate per target
# =========================

all_results = []

for target in available_targets:
    print("\n" + "=" * 80)
    print(f"Side-channel attack target: {target}")
    print("=" * 80)

    df = merged_df[available_features + [target]].copy()
    df = df.dropna(subset=[target])

    # Convert target to string classification labels
    df[target] = df[target].astype(str)

    # Remove classes with fewer than 2 samples
    class_counts = df[target].value_counts()
    valid_classes = class_counts[class_counts >= 2].index
    df = df[df[target].isin(valid_classes)]

    print("Class distribution:")
    print(df[target].value_counts())

    if df[target].nunique() < 2:
        print(f"Skipping {target}: fewer than 2 valid classes.")
        continue

    X = df[available_features]
    y = df[target]

    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average="macro")
        weighted_f1 = f1_score(y_test, y_pred, average="weighted")

        all_results.append({
            "target": target,
            "model": model_name,
            "accuracy": acc,
            "macro_f1": macro_f1,
            #"weighted_f1": weighted_f1,
            #"num_classes": y.nunique(),
            #"num_samples": len(df)
        })

        print(f"\nModel: {model_name}")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")
        print(f"Weighted F1: {weighted_f1:.4f}")
        print(classification_report(y_test, y_pred, zero_division=0))


# =========================
# 5. Save results
# =========================

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(
    by=["target", "macro_f1"],
    ascending=[True, False]
)

print("\nFinal model comparison:")
print(results_df)

results_df.to_excel("side_channel_classifier_results.xlsx", index=False)

print("\nSaved: side_channel_classifier_results.xlsx")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import pandas as pd


targets = ["circuit_type", "n_qubits", "n_layers", "model_type"]

for target in targets:
    print(target)



    #target = "n_qubits"

    df = merged_df[available_features + [target]].copy()
    df = df.dropna(subset=[target])
    df[target] = df[target].astype(str)

    X = df[available_features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    rf_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            class_weight="balanced"
        ))
    ])

    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)

    print(classification_report(y_test, y_pred, zero_division=0))

    rf = rf_model.named_steps["clf"]

    importance_df = pd.DataFrame({
        "feature": available_features,
        "importance": rf.feature_importances_
    }).sort_values("importance", ascending=False)

    print(importance_df)

    importance_df.to_excel(f"feature_importance_{target}.xlsx", index=False)

# Final

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# 1. Define side-channel features and attack targets
# ============================================================

side_channel_features = [
    "execution_time_sec",
    "cpu_energy_kwh",
    "gpu_energy_kwh",
    "ram_energy_kwh",
    "total_energy_kwh",
    "total_emissions_kg",
    "joules_per_token",
    "energy_per_token_kwh",
    "watts_estimated",
    "tokens_per_second",
    "cpu_usage_pct",
    "cpu_clock_mhz",
    "cpu_cores_used",
    "ram_usage_pct",
    "memory_footprint_mb"
]

targets = [
    "circuit_type",
    "n_qubits",
    "n_layers",
    # "model_type"
]


# ============================================================
# 2. Check available columns
# ============================================================

available_features = [
    col for col in side_channel_features
    if col in merged_df.columns
]

available_targets = [
    col for col in targets
    if col in merged_df.columns
]

print("Available side-channel features:")
print(available_features)

print("\nAvailable target columns:")
print(available_targets)

if len(available_features) == 0:
    raise ValueError("No side-channel feature columns found in merged_df.")

if len(available_targets) == 0:
    raise ValueError("No target columns found in merged_df.")


# ============================================================
# 3. Define classifiers
# ============================================================

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]),

    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=True,
            random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(
            n_neighbors=5
        ))
    ])
}


# ============================================================
# 4. Train and evaluate each model for each target
# ============================================================

all_results = []
all_class_reports = []

for target in available_targets:
    print("\n" + "=" * 90)
    print(f"Side-channel attack target: {target}")
    print("=" * 90)

    # Keep only selected side-channel features and one target
    df = merged_df[available_features + [target]].copy()

    # Remove missing target rows
    df = df.dropna(subset=[target])

    # Target as classification label
    df[target] = df[target].astype(str)

    # Remove classes with fewer than 2 samples
    class_counts = df[target].value_counts()
    valid_classes = class_counts[class_counts >= 2].index
    df = df[df[target].isin(valid_classes)]

    print("\nClass distribution:")
    print(df[target].value_counts())

    # Skip if target has fewer than 2 valid classes
    if df[target].nunique() < 2:
        print(f"\nSkipping {target}: fewer than 2 valid classes.")
        continue

    X = df[available_features]
    y = df[target]

    # Check smallest class count
    min_class_count = y.value_counts().min()

    if min_class_count < 2:
        print(f"\nSkipping {target}: at least one class has fewer than 2 samples.")
        continue

    # Stratified train-test split
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
            stratify=y
        )
    except ValueError as e:
        print(f"\nCould not split target {target}: {e}")
        continue

    print("\nTrain size:", len(X_train))
    print("Test size:", len(X_test))

    # Random baseline
    num_classes = y.nunique()
    random_baseline = 1 / num_classes

    print(f"Number of classes: {num_classes}")
    print(f"Random baseline accuracy: {random_baseline:.4f}")

    for model_name, model in models.items():
        print("\n" + "-" * 70)
        print(f"Training model: {model_name}")

        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            acc = accuracy_score(y_test, y_pred)

            macro_precision = precision_score(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            )

            macro_recall = recall_score(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            )

            macro_f1 = f1_score(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            )

            weighted_precision = precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )

            weighted_recall = recall_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )

            weighted_f1 = f1_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            )

            leakage_gain_over_random = acc - random_baseline

            all_results.append({
                "target": target,
                "model": model_name,
                "accuracy": acc,
                "macro_precision": macro_precision,
                "macro_recall": macro_recall,
                "macro_f1": macro_f1,
                # "weighted_precision": weighted_precision,
                # "weighted_recall": weighted_recall,
                # "weighted_f1": weighted_f1,
                # "num_classes": num_classes,
                # "num_samples": len(df),
                # "train_samples": len(X_train),
                # "test_samples": len(X_test),
                # "random_baseline_accuracy": random_baseline,
                #"leakage_gain_over_random": leakage_gain_over_random
            })

            print(f"Accuracy: {acc:.4f}")
            print(f"Macro Precision: {macro_precision:.4f}")
            print(f"Macro Recall: {macro_recall:.4f}")
            print(f"Macro F1: {macro_f1:.4f}")
            print(f"Weighted Precision: {weighted_precision:.4f}")
            print(f"Weighted Recall: {weighted_recall:.4f}")
            print(f"Weighted F1: {weighted_f1:.4f}")
            print(f"Leakage gain over random baseline: {leakage_gain_over_random:.4f}")

            print("\nClassification report:")
            print(classification_report(y_test, y_pred, zero_division=0))

            # Save detailed classification report as dataframe
            report_dict = classification_report(
                y_test,
                y_pred,
                output_dict=True,
                zero_division=0
            )

            report_df = pd.DataFrame(report_dict).transpose()
            report_df["target"] = target
            report_df["model"] = model_name

            all_class_reports.append(report_df)

        except Exception as e:
            print(f"Model failed: {model_name}")
            print("Error:", e)


# ============================================================
# 5. Save summary results
# ============================================================

results_df = pd.DataFrame(all_results)

if not results_df.empty:
    results_df = results_df.sort_values(
        by=["target", "macro_f1"],
        ascending=[True, False]
    )

    print("\n" + "=" * 90)
    print("Final model comparison:")
    print("=" * 90)
    print(results_df)

    results_df.to_excel(
        "side_channel_classifier_results.xlsx",
        index=False
    )

    print("\nSaved summary results as: side_channel_classifier_results.xlsx")

else:
    print("\nNo model results were generated.")


# ============================================================
# 6. Save detailed classification reports
# ============================================================

if len(all_class_reports) > 0:
    class_report_df = pd.concat(all_class_reports, axis=0)

    # Move target/model to front
    cols = ["target", "model"] + [
        col for col in class_report_df.columns
        if col not in ["target", "model"]
    ]

    class_report_df = class_report_df[cols]

    class_report_df.to_excel(
        "side_channel_classification_reports.xlsx",
        index=True
    )

    print("Saved detailed classification reports as: side_channel_classification_reports.xlsx")
else:
    print("No detailed classification reports were generated.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. Define side-channel features
# ============================================================

side_channel_features = [
    "execution_time_sec",
    "cpu_energy_kwh",
    "gpu_energy_kwh",
    "ram_energy_kwh",
    "total_energy_kwh",
    "total_emissions_kg",
    "joules_per_token",
    "energy_per_token_kwh",
    "watts_estimated",
    "tokens_per_second",
    "cpu_usage_pct",
    "cpu_clock_mhz",
    "cpu_cores_used",
    "ram_usage_pct",
    "memory_footprint_mb"
]

# Keep only available columns
available_features = [col for col in side_channel_features if col in merged_df.columns]

print("Available side-channel features:")
print(available_features)

# ============================================================
# 2. Histograms for each feature
# ============================================================

for feature in available_features:
    plt.figure(figsize=(8, 5))
    merged_df[feature].dropna().plot(kind="hist", bins=30)
    plt.title(f"Histogram of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

# ============================================================
# 3. Boxplots for each feature
# ============================================================

for feature in available_features:
    plt.figure(figsize=(6, 5))
    plt.boxplot(merged_df[feature].dropna())
    plt.title(f"Boxplot of {feature}")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

# ============================================================
# 4. Correlation heatmap
# ============================================================

corr_df = merged_df[available_features].corr()

plt.figure(figsize=(12, 10))
plt.imshow(corr_df, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90)
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.title("Correlation Heatmap of Side-Channel Features")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# importance_df must have columns: feature, importance
df = importance_df.copy()

df = df.sort_values("importance", ascending=False).reset_index(drop=True)

max_val = df["importance"].max()

def get_tier_and_color(value, max_value):
    ratio = value / max_value

    if ratio >= 0.75:
        return "≥ 75% of max : Very High", "#c62828"
    elif ratio >= 0.50:
        return "50–75% : High", "#ef6c00"
    elif ratio >= 0.25:
        return "25–50% : Moderate", "#f9a825"
    elif ratio >= 0.10:
        return "10–25% : Low", "#558b2f"
    else:
        return "< 10% : Very Low", "#1b5e20"

tiers = df["importance"].apply(lambda x: get_tier_and_color(x, max_val))
df["tier"] = tiers.apply(lambda x: x[0])
df["color"] = tiers.apply(lambda x: x[1])



fig, ax = plt.subplots(figsize=(10, 10))
#plt.figure(figsize=(10, 10))
#plt.figure(figsize=(3.5, 2.8))
plt.barh(df["feature"], df["importance"], color=df["color"])
plt.gca().invert_yaxis()

for i, value in enumerate(df["importance"]):
    plt.text(value + max_val * 0.01, i, f"{value:.5f}", va="center", fontsize=10)

#plt.title("Side-Channel Feature Saliency", fontsize=16, weight="bold")
plt.xlabel("Feature Importance / Saliency Score", fontsize=12)
plt.ylabel("")
plt.grid(axis="x", alpha=0.25)
plt.gca().set_axisbelow(True)

legend_elements = [
    Patch(facecolor="#c62828", label="≥ 75% of max : Very High"),
    Patch(facecolor="#ef6c00", label="50–75%       : High"),
    Patch(facecolor="#f9a825", label="25–50%       : Moderate"),
    Patch(facecolor="#558b2f", label="10–25%       : Low"),
    Patch(facecolor="#1b5e20", label="< 10%        : Very Low")
]

# Remove top bar/spine
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.legend(handles=legend_elements, title="Attention tier", loc="lower right")
plt.tight_layout()
plt.savefig("side_channel_feature_saliency.png", dpi=300, bbox_inches="tight")
plt.show()

In [37]:
# ============================================================
# BERT-ONLY SIDE-CHANNEL CLASSIFIER - DEBUG FIXED VERSION
# ============================================================

# If needed:
# pip install pandas numpy scikit-learn transformers torch accelerate openpyxl

import pandas as pd
import numpy as np
import torch
import traceback

from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)


# ============================================================
# 1. Define features and targets
# ============================================================

side_channel_features = [
    "execution_time_sec",
    "cpu_energy_kwh",
    "gpu_energy_kwh",
    "ram_energy_kwh",
    "total_energy_kwh",
    "total_emissions_kg",
    "joules_per_token",
    "energy_per_token_kwh",
    "watts_estimated",
    "tokens_per_second",
    "cpu_usage_pct",
    "cpu_clock_mhz",
    "cpu_cores_used",
    "ram_usage_pct",
    "memory_footprint_mb"
]

targets = [
    "circuit_type",
    "n_qubits",
    "n_layers",
    # "model_type"
]


# ============================================================
# 2. Check merged_df
# ============================================================

if "merged_df" not in globals():
    raise ValueError("merged_df is not found. Please load your merged dataset first.")

available_features = [
    col for col in side_channel_features
    if col in merged_df.columns
]

available_targets = [
    col for col in targets
    if col in merged_df.columns
]

print("Available side-channel features:")
print(available_features)

print("\nAvailable targets:")
print(available_targets)

if len(available_features) == 0:
    raise ValueError("No side-channel features found in merged_df.")

if len(available_targets) == 0:
    raise ValueError("No target columns found in merged_df.")


# ============================================================
# 3. Helper functions
# ============================================================

def row_to_text(row, feature_cols):
    parts = []

    for col in feature_cols:
        value = row[col]

        if pd.isna(value):
            value_text = "missing"
        else:
            try:
                value_text = f"{float(value):.10g}"
            except Exception:
                value_text = str(value)

        parts.append(f"{col} is {value_text}")

    return ", ".join(parts)


class SideChannelTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


def compute_metrics_from_labels(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "weighted_precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "weighted_recall": recall_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
    }


def compute_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision_score(
            labels, preds, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            labels, preds, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            labels, preds, average="macro", zero_division=0
        )
    }


# ============================================================
# 4. BERT training function
# ============================================================

def train_bert_side_channel_classifier(
    merged_df,
    feature_cols,
    target,
    bert_model_name="distilbert-base-uncased",
    test_size=0.2,
    random_state=42,
    num_train_epochs=3,
    batch_size=8,
    max_length=256
):
    print("\n" + "=" * 90)
    print(f"BERT side-channel target: {target}")
    print("=" * 90)

    df = merged_df[feature_cols + [target]].copy()
    df = df.dropna(subset=[target])

    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

        if df[col].isna().all():
            df[col] = 0
        else:
            df[col] = df[col].fillna(df[col].median())

    df[target] = df[target].astype(str)

    print("\nOriginal class distribution:")
    print(df[target].value_counts())

    # IMPORTANT:
    # For stratified 80/20 split, each class should ideally have at least 5 samples.
    # Otherwise the test set may not contain every class.
    class_counts = df[target].value_counts()
    valid_classes = class_counts[class_counts >= 5].index
    df = df[df[target].isin(valid_classes)].copy()

    print("\nClass distribution after removing classes with fewer than 5 samples:")
    print(df[target].value_counts())

    if df[target].nunique() < 2:
        print(f"Skipping {target}: fewer than 2 valid classes after filtering.")
        return None, None, None

    df["bert_text"] = df.apply(
        lambda row: row_to_text(row, feature_cols),
        axis=1
    )

    label_encoder = LabelEncoder()
    df["label_id"] = label_encoder.fit_transform(df[target])

    X_text = df["bert_text"]
    y = df["label_id"]

    num_classes = df["label_id"].nunique()
    random_baseline_accuracy = 1 / num_classes
    majority_baseline_accuracy = df[target].value_counts(normalize=True).max()

    print(f"\nNumber of classes: {num_classes}")
    print(f"Random baseline accuracy: {random_baseline_accuracy:.4f}")
    print(f"Majority baseline accuracy: {majority_baseline_accuracy:.4f}")

    X_train, X_test, y_train, y_test = train_test_split(
        X_text,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print("Train size:", len(X_train))
    print("Test size:", len(X_test))

    tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

    train_dataset = SideChannelTextDataset(
        texts=X_train,
        labels=y_train,
        tokenizer=tokenizer,
        max_length=max_length
    )

    test_dataset = SideChannelTextDataset(
        texts=X_test,
        labels=y_test,
        tokenizer=tokenizer,
        max_length=max_length
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        bert_model_name,
        num_labels=num_classes
    )

    # Compatibility fix:
    # Some transformers versions use evaluation_strategy.
    # Newer versions may use eval_strategy.
    try:
        training_args = TrainingArguments(
            output_dir=f"./bert_side_channel_{target}",
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=2e-5,
            weight_decay=0.01,
            logging_steps=20,
            evaluation_strategy="epoch",
            save_strategy="no",
            report_to="none",
            load_best_model_at_end=False
        )
    except TypeError:
        training_args = TrainingArguments(
            output_dir=f"./bert_side_channel_{target}",
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=2e-5,
            weight_decay=0.01,
            logging_steps=20,
            eval_strategy="epoch",
            save_strategy="no",
            report_to="none",
            load_best_model_at_end=False
        )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_trainer_metrics
    )

    trainer.train()

    predictions = trainer.predict(test_dataset)

    logits = predictions.predictions
    pred_ids = np.argmax(logits, axis=1)

    probs = torch.softmax(
        torch.tensor(logits),
        dim=1
    ).numpy()

    y_test_array = np.array(y_test)

    y_test_labels = label_encoder.inverse_transform(y_test_array)
    y_pred_labels = label_encoder.inverse_transform(pred_ids)

    metrics = compute_metrics_from_labels(
        y_true=y_test_labels,
        y_pred=y_pred_labels
    )

    leakage_gain_over_random = (
        metrics["accuracy"] - random_baseline_accuracy
    )

    leakage_gain_over_majority = (
        metrics["accuracy"] - majority_baseline_accuracy
    )

    print("\nBERT results:")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Macro Precision: {metrics['macro_precision']:.4f}")
    print(f"Macro Recall: {metrics['macro_recall']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
    print(f"Leakage gain over random: {leakage_gain_over_random:.4f}")
    print(f"Leakage gain over majority: {leakage_gain_over_majority:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_test_labels,
            y_pred_labels,
            zero_division=0
        )
    )

    summary_result = {
        "target": target,
        "model": f"BERT ({bert_model_name})",
        "accuracy": metrics["accuracy"],
        "macro_precision": metrics["macro_precision"],
        "macro_recall": metrics["macro_recall"],
        "macro_f1": metrics["macro_f1"],
        "weighted_precision": metrics["weighted_precision"],
        "weighted_recall": metrics["weighted_recall"],
        "weighted_f1": metrics["weighted_f1"],
        "num_classes": num_classes,
        "num_samples": len(df),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "random_baseline_accuracy": random_baseline_accuracy,
        "majority_baseline_accuracy": majority_baseline_accuracy,
        "leakage_gain_over_random": leakage_gain_over_random,
        "leakage_gain_over_majority": leakage_gain_over_majority
    }

    report_dict = classification_report(
        y_test_labels,
        y_pred_labels,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()
    report_df["target"] = target
    report_df["model"] = f"BERT ({bert_model_name})"

    prediction_rows = []
    test_texts = list(X_test)

    for i in range(len(test_texts)):
        row_data = {
            "target": target,
            "bert_model": bert_model_name,
            "text_input": test_texts[i],
            "true_label": y_test_labels[i],
            "predicted_label": y_pred_labels[i],
            "correct": y_test_labels[i] == y_pred_labels[i],
            "max_probability": float(probs[i].max())
        }

        for class_idx, class_name in enumerate(label_encoder.classes_):
            row_data[f"prob_{class_name}"] = float(probs[i][class_idx])

        prediction_rows.append(row_data)

    prediction_df = pd.DataFrame(prediction_rows)

    return summary_result, report_df, prediction_df


# ============================================================
# 5. Run BERT only
# ============================================================

all_results = []
all_class_reports = []
all_prediction_details = []

BERT_MODEL_NAME = "distilbert-base-uncased"

for target in available_targets:
    try:
        summary_result, report_df, prediction_df = train_bert_side_channel_classifier(
            merged_df=merged_df,
            feature_cols=available_features,
            target=target,
            bert_model_name=BERT_MODEL_NAME,
            test_size=0.2,
            random_state=42,
            num_train_epochs=3,
            batch_size=8,
            max_length=256
        )

        if summary_result is not None:
            all_results.append(summary_result)

        if report_df is not None:
            all_class_reports.append(report_df)

        if prediction_df is not None:
            all_prediction_details.append(prediction_df)

    except Exception as e:
        print("\n" + "!" * 90)
        print(f"BERT failed for target: {target}")
        print("Error:")
        print(e)
        print("\nFull traceback:")
        traceback.print_exc()
        print("!" * 90)


# ============================================================
# 6. Save BERT summary results
# ============================================================

bert_results_df = pd.DataFrame(all_results)

if not bert_results_df.empty:
    bert_results_df = bert_results_df.sort_values(
        by=["target", "macro_f1"],
        ascending=[True, False]
    )

    print("\nFinal BERT-only summary results:")
    print(bert_results_df)

    bert_results_df.to_excel(
        "bert_side_channel_summary_results.xlsx",
        index=False
    )

    print("\nSaved: bert_side_channel_summary_results.xlsx")
else:
    print("\nNo BERT summary results were generated.")


# ============================================================
# 7. Save BERT classification reports
# ============================================================

if len(all_class_reports) > 0:
    bert_class_report_df = pd.concat(
        all_class_reports,
        axis=0
    )

    cols = ["target", "model"] + [
        col for col in bert_class_report_df.columns
        if col not in ["target", "model"]
    ]

    bert_class_report_df = bert_class_report_df[cols]

    bert_class_report_df.to_excel(
        "bert_side_channel_classification_reports.xlsx",
        index=True
    )

    print("Saved: bert_side_channel_classification_reports.xlsx")
else:
    print("No BERT classification reports were generated.")


# ============================================================
# 8. Save BERT prediction details
# ============================================================

if len(all_prediction_details) > 0:
    bert_prediction_details_df = pd.concat(
        all_prediction_details,
        ignore_index=True
    )

    bert_prediction_details_df.to_excel(
        "bert_side_channel_prediction_details.xlsx",
        index=False
    )

    print("Saved: bert_side_channel_prediction_details.xlsx")
else:
    print("No BERT prediction details were generated.")

Available side-channel features:
['execution_time_sec', 'cpu_energy_kwh', 'gpu_energy_kwh', 'ram_energy_kwh', 'total_energy_kwh', 'total_emissions_kg', 'joules_per_token', 'energy_per_token_kwh', 'watts_estimated', 'tokens_per_second', 'cpu_usage_pct', 'cpu_clock_mhz', 'cpu_cores_used', 'ram_usage_pct', 'memory_footprint_mb']

Available targets:
['circuit_type', 'n_qubits', 'n_layers']

BERT side-channel target: circuit_type

Original class distribution:
circuit_type
RY_RZ_LINEAR             2880
RX_RY_RING               2880
HARDWARE_EFFICIENT_CZ    2684
DATA_REUPLOAD            2520
Name: count, dtype: int64

Class distribution after removing classes with fewer than 5 samples:
circuit_type
RY_RZ_LINEAR             2880
RX_RY_RING               2880
HARDWARE_EFFICIENT_CZ    2684
DATA_REUPLOAD            2520
Name: count, dtype: int64

Number of classes: 4
Random baseline accuracy: 0.2500
Majority baseline accuracy: 0.2627
Train size: 8771
Test size: 2193


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6188.94it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Traceback (most recent call last):
  File "C:\Users\SIU856536670\AppData\Local\Temp\ipykernel_32948\3177390881.py", line 288, in train_bert_side_chann


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
BERT failed for target: circuit_type
Error:
Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

Full traceback:
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

BERT side-channel target: n_qubits

Original class distribution:
n_qubits
1    2232
2    2232
3    2180
4    2160
5    2160
Name: count, dtype: int64

Class distribution after removing classes with fewer than 5 samples:
n_qubits
1    2232
2    2232
3    2180
4    2160
5    2160
Name: count, dtype: int64

Number of classes: 5
Random baseline accuracy: 0.2000
Majority baseline accuracy: 0.2036
Train size: 8771
Test size: 2193


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5843.77it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
BERT failed for target: n_qubits
Error:
Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

Full traceback:
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

BERT side-channel target: n_layers

Original class distribution:
n_layers
4    10964
Name: count, dtype: int64

Class distribution after removing classes with fewer than 5 samples:
n_layers
4    10964
Name: count, dtype: int64
Skipping n_layers: fewer than 2 valid classes after filtering.

No BERT summary results were generated.
No BERT classification reports were generated.
No BERT prediction details were generated.


Traceback (most recent call last):
  File "C:\Users\SIU856536670\AppData\Local\Temp\ipykernel_32948\3177390881.py", line 288, in train_bert_side_channel_classifier
    training_args = TrainingArguments(
TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\SIU856536670\AppData\Local\Temp\ipykernel_32948\3177390881.py", line 439, in <module>
    summary_result, report_df, prediction_df = train_bert_side_channel_classifier(
  File "C:\Users\SIU856536670\AppData\Local\Temp\ipykernel_32948\3177390881.py", line 302, in train_bert_side_channel_classifier
    training_args = TrainingArguments(
  File "<string>", line 115, in __init__
  File "c:\Users\SIU856536670\AppData\Local\anaconda3\envs\qantum\lib\site-packages\transformers\training_args.py", line 1599, in __post_init__
    self.device
  File "c:\Users\SIU856536670\AppData\Lo